# E-Commerce Lakehouse: Silver Order Items

## Convert source data types

Read the saved Bronze order items and convert:
- order_item_id to an integer.
- shipping_limit_date to a timestamp.
- price and freight_value to decimal values with two decimal places.

Preserve identifiers as strings and retain ingestion metadata.
Validate conversions before saving the Silver table.

In [0]:
from pyspark.sql.functions import col, to_timestamp

bronze_order_items_df = spark.table(
    "workspace.ecommerce.bronze_order_items"
)

silver_order_items_df = (
    bronze_order_items_df
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn(
        "shipping_limit_date",
        to_timestamp("shipping_limit_date", "yyyy-MM-dd HH:mm:ss")
    )
    .withColumn("price", col("price").cast("decimal(12,2)"))
    .withColumn("freight_value", col("freight_value").cast("decimal(12,2)"))
)

silver_order_items_df.printSchema()
display(silver_order_items_df.limit(5))

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: decimal(12,2) (nullable = true)
 |-- freight_value: decimal(12,2) (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.90,13.29,2026-10-17T20:17:51.704Z
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.90,19.93,2026-10-17T20:17:51.704Z
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.00,17.87,2026-10-17T20:17:51.704Z
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18.000Z,12.99,12.79,2026-10-17T20:17:51.704Z
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51.000Z,199.90,18.14,2026-10-17T20:17:51.704Z


## Validate Silver order items

Compare source and Silver null counts for converted columns.
Check required identifiers, duplicate order-item keys, and invalid
numeric values. Preserve records while investigating any issues.

In [0]:
from pyspark.sql.functions import trim

print("Bronze rows:", bronze_order_items_df.count())
print("Silver rows:", silver_order_items_df.count())

# Check whether conversions introduced additional nulls.
for column_name in [
    "order_item_id",
    "shipping_limit_date",
    "price",
    "freight_value"
]:
    bronze_nulls = bronze_order_items_df.filter(
        col(column_name).isNull()
    ).count()

    silver_nulls = silver_order_items_df.filter(
        col(column_name).isNull()
    ).count()

    print(
        f"{column_name}: "
        f"Bronze nulls={bronze_nulls}, Silver nulls={silver_nulls}"
    )

# Check required text identifiers.
for column_name in ["order_id", "product_id", "seller_id"]:
    missing_count = silver_order_items_df.filter(
        col(column_name).isNull()
        | (trim(col(column_name)) == "")
    ).count()

    print(f"Missing or blank {column_name}: {missing_count}")

# An order can have multiple items, but each item key should be unique.
duplicate_item_keys_df = (
    silver_order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter(col("count") > 1)
)

print("Duplicate order-item keys:", duplicate_item_keys_df.count())

# Zero freight is allowed; negative amounts are flagged.
invalid_values_df = silver_order_items_df.filter(
    (col("order_item_id") <= 0)
    | (col("price") < 0)
    | (col("freight_value") < 0)
)

print("Rows with invalid numeric values:", invalid_values_df.count())

Bronze rows: 112650
Silver rows: 112650
order_item_id: Bronze nulls=0, Silver nulls=0
shipping_limit_date: Bronze nulls=0, Silver nulls=0
price: Bronze nulls=0, Silver nulls=0
freight_value: Bronze nulls=0, Silver nulls=0
Missing or blank order_id: 0
Missing or blank product_id: 0
Missing or blank seller_id: 0
Duplicate order-item keys: 0
Rows with invalid numeric values: 0


## Validate order relationships and save Silver order items

Check that every order item references an existing Silver order.
Save the validated data as a managed Delta table and verify
the stored row count. Leave an existing table unchanged.

In [0]:
silver_orders_df = spark.table("workspace.ecommerce.silver_orders")

unmatched_count = silver_order_items_df.join(
    silver_orders_df.select("order_id"),
    on="order_id",
    how="left_anti"
).count()

row_count = silver_order_items_df.count()

duplicate_key_count = (
    silver_order_items_df
    .groupBy("order_id", "order_item_id")
    .count()
    .filter("count > 1")
    .count()
)

print("Items without a matching order:", unmatched_count)
print("Silver item rows:", row_count)
print("Duplicate order-item keys:", duplicate_key_count)

if (
    unmatched_count != 0
    or row_count != 112650
    or duplicate_key_count != 0
):
    raise ValueError(
        "Silver item validation failed. Stopped before replacing the table."
    )

table_name = "workspace.ecommerce.silver_order_items"

(
    silver_order_items_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

saved_silver_order_items_df = spark.table(table_name)
saved_count = saved_silver_order_items_df.count()

if saved_count != row_count:
    raise ValueError("Saved Silver item count does not match.")

print("Silver order items refreshed:", saved_count)

Items without a matching order: 0
Silver item rows: 112650
Duplicate order-item keys: 0
Silver order items refreshed: 112650
